# Schrödinger equation: Crank–Nicolson time stepping

*quantum · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/quantum/schrodinger_crank_nicolson)

The time-dependent Schrödinger equation $i\,\partial_t\psi = H\psi$ is marched on a grid with the implicit Crank–Nicolson scheme $(I + \tfrac{i\Delta t}{2}H)\,\psi^{n+1} = (I - \tfrac{i\Delta t}{2}H)\,\psi^n$. The left matrix is LU-factorised once and every step is a cheap torch.linalg.lu_solve for three wave packets at once (the columns of $\psi_{xb}$). The step is exactly unitary, so the norm stays 1 while explicit Euler with the same $\Delta t$ explodes.

**Vector form**

$$i\,\partial_t\psi = \Big(-\tfrac12\,\partial_x^2 + \tfrac12 x^2 + g\,x^4\Big)\psi,\qquad \Big(1 + \frac{i\Delta t}{2}\hat H\Big)\psi^{n+1} = \Big(1 - \frac{i\Delta t}{2}\hat H\Big)\psi^{n}$$

**Index (tensor) form**

$$H_{xy} = -\tfrac12\,D2_{xy} + V_x\,\delta_{xy},\qquad A_{xy} = \delta_{xy} + \tfrac{i\Delta t}{2}H_{xy},\qquad A_{xy}\,\psi^{n+1}_{yb} = \big(\delta_{xy} - \tfrac{i\Delta t}{2}H_{xy}\big)\,\psi^{n}_{yb}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_x$ | `potential(x, g)` | (N,) | anharmonic well $\tfrac12x^2 + g\,x^4$: the oscillation period depends on the amplitude |
| $H_{xy}$ | `hamiltonian(x, dx, V)` | (N, N) | kinetic $-\tfrac12\,D2$ plus $V$ on the diagonal; walls at the box edges |
| $\psi^0_{xb}$ | `packets(x, x0s)` | (N, B) complex | one Gaussian per column $b$, released at rest from $x_{0,b}$ |
| $A_{xy},\ B_{xy}$ | `cayley_matrices(H, dt)` | (N, N), (N, N) | $I \pm \tfrac{i\Delta t}{2}H$, built once before the loop |
| $A = PLU$ | `torch.linalg.lu_factor(A)` | (N, N), (N,) | factorise once ($\mathcal O(N^3)$); each step then costs only $\mathcal O(N^2)$ |
| $\psi^{n+1}_{xb}$ | `lu_solve(lu, piv, einsum('xy,yb->xb', B, psi))` | (N, B) complex | repeated $y$ ⇒ summed: right-hand side, then two triangular solves for all columns |
| $\psi^{n+1} = (I - i\Delta t H)\psi^n$ | `euler_step(H, psi, dt)` | (N, B) complex | explicit Euler for comparison: each eigen-component grows by $\sqrt{1 + E^2\Delta t^2}$ per step |
| $\langle H\rangle_b$ | `einsum('xb,xy,yb->b', psi.conj(), H, psi) * dx` | (B,) | repeated $x, y$ ⇒ summed: energy of each packet, conserved exactly |

Units $\hbar = m = 1$. Why it works: $U = (I + \tfrac{i\Delta t}{2}H)^{-1}(I - \tfrac{i\Delta t}{2}H)$ is the Cayley transform of the Hermitian matrix $H$. On an eigenvector with energy $E$ it multiplies by $(1 - iE\Delta t/2)/(1 + iE\Delta t/2)$, a number of modulus exactly 1, so $U$ is unitary for any $\Delta t$: no growth, no decay, and because $U$ is a function of $H$, $\langle H\rangle$ is conserved too. Its phase $2\arctan(E\Delta t/2)$ matches the exact $E\Delta t$ up to $\mathcal O(E^3\Delta t^3)$ per step, so the scheme is second-order accurate, and being implicit it is stable for any $\Delta t$ (large steps are only less accurate, never unstable). Explicit Euler instead multiplies by $1 - iE\Delta t$, of modulus $\sqrt{1 + E^2\Delta t^2} > 1$; the stiff grid modes with $E \sim 2/dx^2$ blow up within a few dozen steps (red curve). The three packets have different amplitudes; the stiffer $g\,x^4$ spring makes the big one swing faster, and because its energy levels are no longer evenly spaced it dephases — spreads out and loses its shape — while the small packet barely feels the anharmonicity. Try $g = 0$: all three swing together with period $2\pi$ and never spread (coherent states). Compare with the split-step FFT in Quantum tunnelling, the other unitary second-order scheme.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/quantum/schrodinger_crank_nicolson/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import energy, mean_position, norm


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    g, x0, dt = params["g"], params["x0"], params["dt"]
    x0s = [x0, 2 * x0 / 3, x0 / 3]  # three packets released from different heights, evolved as columns
    x, dx = tensors.grid(params["n_grid"], 2 * (x0 + 5), dev)
    V = tensors.potential(x, g)
    H = tensors.hamiltonian(x, dx, V)
    psi0 = tensors.packets(x, x0s)  # (N, B)

    times, psi = solve.crank_nicolson(H, psi0, dt, params["t_max"])  # (T,), (T, N, B)
    t_euler, n_euler = solve.euler_norms(H, psi0, dx, dt, params["t_max"])  # (S,), (S, B)

    norms, energies = norm(psi, dx), energy(psi, H, dx)  # (T, B) each
    print(f"grid N = {len(x)}, dx = {dx:.4f}, dt = {dt}, {round(params['t_max'] / dt)} Crank–Nicolson steps")
    for b, start in enumerate(x0s):
        print(
            f"packet x0 = {start:4.2f}: <H> = {energies[0, b].item():8.4f} -> {energies[-1, b].item():8.4f}   "
            f"|norm - 1| max = {(norms[:, b] - 1).abs().max().item():.1e}"
        )
    print(f"explicit Euler, same dt: norm = {n_euler[-1].max().item():.1e} after {len(t_euler) - 1} steps")
    err = (psi[-1] - solve.exact(H, psi0, times[-1].item())).abs().max().item()
    print(f"max |psi_CN - exp(-iHt) psi0| at t_max = {err:.2e}   (shrinks like dt^2)")
    x_mean = mean_position(psi, x, dx)  # (T, B)
    return plot.animate(
        x, V, times, psi, x_mean, norms, energies, t_euler, n_euler, x0s, dt, out_dir / "schrodinger_crank_nicolson.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/quantum/schrodinger_crank_nicolson/equation.py
r"""Physics → tensor translation.

Time-dependent Schrödinger equation (units ħ = m = 1) on N grid points:

    i \partial_t \psi_x = H_{xy} \psi_y,        H_{xy} = -\tfrac12 D2_{xy} + V_x \delta_{xy}      (y summed).

Several wave packets are evolved at once as the columns b of \psi_{xb}.

Crank–Nicolson averages the right-hand side over the old and the new time level:

    i (\psi^{n+1} - \psi^n)/\Delta t = \tfrac12 H (\psi^{n+1} + \psi^n)
    ⇔  A_{xy} \psi^{n+1}_{yb} = B_{xy} \psi^n_{yb},
        A = I + \tfrac{i\Delta t}{2} H,   B = I - \tfrac{i\Delta t}{2} H       (y summed).

U = A^{-1} B is the Cayley transform of the Hermitian H, which is exactly unitary, so the norm never drifts.
For comparison, explicit Euler takes \psi^{n+1} = (I - i\Delta t H)\psi^n, whose eigenvalues
1 - iE\Delta t all have modulus \sqrt{1 + E^2\Delta t^2} > 1: the norm grows every step.

    norm_b   = \psi^*_{xb} \psi_{xb} dx               (x summed)
    \langle H\rangle_b = \psi^*_{xb} H_{xy} \psi_{yb} dx     (x, y summed)
"""

import torch


def cayley_matrices(H: torch.Tensor, dt: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""A = I + i\Delta t H/2,  B = I - i\Delta t H/2.   H: (N, N) real → two (N, N) complex matrices."""
    eye = torch.eye(len(H), dtype=torch.complex128, device=H.device)
    Hc = H.to(torch.complex128)
    return eye + 0.5j * dt * Hc, eye - 0.5j * dt * Hc


def apply(M: torch.Tensor, psi: torch.Tensor) -> torch.Tensor:
    r"""M_{xy} \psi_{yb}  (y summed).   M: (N, N), psi: (N, B) → (N, B)"""
    return torch.einsum("xy,yb->xb", M, psi)


def cn_step(lu: torch.Tensor, pivots: torch.Tensor, B: torch.Tensor, psi: torch.Tensor) -> torch.Tensor:
    r"""Solve A \psi^{n+1} = B \psi^n with the LU factors of A (computed once).   (N, B) → (N, B)"""
    return torch.linalg.lu_solve(lu, pivots, apply(B, psi))


def cn_step_solve(A: torch.Tensor, B: torch.Tensor, psi: torch.Tensor) -> torch.Tensor:
    r"""The same step with a fresh torch.linalg.solve (factorises A again every call).   (N, B) → (N, B)"""
    return torch.linalg.solve(A, apply(B, psi))


def euler_step(H: torch.Tensor, psi: torch.Tensor, dt: float) -> torch.Tensor:
    r"""Explicit Euler: \psi^{n+1} = \psi^n - i\Delta t H \psi^n.   (N, B) → (N, B)"""
    return psi - 1j * dt * apply(H.to(psi.dtype), psi)


def norm(psi: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\psi^*_{xb} \psi_{xb} dx  (x summed).   psi: (..., N, B) → (..., B)"""
    return torch.einsum("...xb,...xb->...b", psi.conj(), psi).real * dx


def energy(psi: torch.Tensor, H: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\langle H\rangle_b = \psi^*_{xb} H_{xy} \psi_{yb} dx  (x, y summed).   psi: (..., N, B) → (..., B)"""
    return torch.einsum("...xb,xy,...yb->...b", psi.conj(), H.to(psi.dtype), psi).real * dx


def mean_position(psi: torch.Tensor, x: torch.Tensor, dx: float) -> torch.Tensor:
    r"""\langle x\rangle_b = \psi^*_{xb} x_x \psi_{xb} dx  (x summed).   psi: (..., N, B) → (..., B)"""
    return torch.einsum("...xb,x,...xb->...b", psi.conj(), x.to(psi.dtype), psi).real * dx


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/quantum/schrodinger_crank_nicolson/tensors.py
"""Build the concrete tensors: the grid, the anharmonic well, the Hamiltonian matrix and the start packets."""

import math

import torch


def grid(n: int, L: float, device: torch.device) -> tuple[torch.Tensor, float]:
    """N interior points of the box [−L/2, L/2]; the walls are left out (ψ = 0 there). Returns (x (N,), dx)."""
    dx = L / (n + 1)
    x = -L / 2 + dx * torch.arange(1, n + 1, dtype=torch.float64, device=device)  # (N,)
    return x, dx


def potential(x: torch.Tensor, g: float) -> torch.Tensor:
    """V_x = ½ x² + g x⁴: a spring that stiffens with amplitude (g = 0 is the plain harmonic well). Shape (N,)."""
    return 0.5 * x**2 + g * x**4


def second_derivative(n: int, dx: float, device: torch.device) -> torch.Tensor:
    """D2_{xy}: −2 on the diagonal, 1 on the two neighbouring diagonals, all over dx². Shape (N, N)."""
    main = -2.0 * torch.ones(n, dtype=torch.float64, device=device)  # (N,)
    side = torch.ones(n - 1, dtype=torch.float64, device=device)  # (N-1,)
    return (torch.diag(main) + torch.diag(side, 1) + torch.diag(side, -1)) / dx**2


def hamiltonian(x: torch.Tensor, dx: float, V: torch.Tensor) -> torch.Tensor:
    """H_{xy} = −½ D2_{xy} + V_x δ_{xy}. Shape (N, N), real symmetric."""
    return -0.5 * second_derivative(len(x), dx, x.device) + torch.diag(V)


def packets(x: torch.Tensor, x0s: list[float]) -> torch.Tensor:
    """ψ_{xb}(0): the harmonic ground-state Gaussian π^{-¼} e^{−(x − x0_b)²/2}, one column per start point.

    Shape (N, B) complex. Each packet starts at rest.
    """
    centres = torch.tensor(x0s, dtype=torch.float64, device=x.device)  # (B,)
    psi = math.pi**-0.25 * torch.exp(-((x[:, None] - centres[None, :]) ** 2) / 2)  # (N, B)
    return psi.to(torch.complex128)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/quantum/schrodinger_crank_nicolson/solve.py
"""March the packets with Crank–Nicolson (LU factorised once); compare with explicit Euler and the exact e^{−iHt}."""

import torch

from problems._common.progress import steps

from .equation import cayley_matrices, cn_step, cn_step_solve, euler_step, norm


def crank_nicolson(
    H: torch.Tensor, psi0: torch.Tensor, dt: float, t_max: float, n_saved: int = 240, use_lu: bool = True
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), ψ (T, N, B) complex), keeping about n_saved snapshots.

    A and B never change, so A is LU-factorised once (O(N³)); every step is then a matrix–vector product
    and two triangular solves (O(N²) per packet). use_lu=False calls torch.linalg.solve every step instead.
    """
    n_steps = max(1, int(round(t_max / dt)))
    save_every = max(1, n_steps // n_saved)
    A, B = cayley_matrices(H, dt)  # (N, N) each
    lu, pivots = torch.linalg.lu_factor(A)

    psi, snapshots, times = psi0, [psi0], [0.0]
    for step in steps(n_steps, "Crank–Nicolson steps"):
        psi = cn_step(lu, pivots, B, psi) if use_lu else cn_step_solve(A, B, psi)
        if (step + 1) % save_every == 0:
            snapshots.append(psi)
            times.append((step + 1) * dt)
    return torch.tensor(times, dtype=torch.float64), torch.stack(snapshots)


def euler_norms(
    H: torch.Tensor, psi0: torch.Tensor, dx: float, dt: float, t_max: float, cap: float = 1e12
) -> tuple[torch.Tensor, torch.Tensor]:
    """Explicit Euler with the same Δt; return (times (S,), norm (S, B)), stopping once the norm passes cap."""
    n_steps = max(1, int(round(t_max / dt)))
    psi, norms, times = psi0, [norm(psi0, dx)], [0.0]
    for step in range(n_steps):
        psi = euler_step(H, psi, dt)
        norms.append(norm(psi, dx))
        times.append((step + 1) * dt)
        if norms[-1].max() > cap:
            break
    return torch.tensor(times, dtype=torch.float64), torch.stack(norms)


def exact(H: torch.Tensor, psi0: torch.Tensor, t: float) -> torch.Tensor:
    """Reference ψ(t) = e^{−iHt} ψ0 = φ e^{−iEt} φᵀ ψ0 from torch.linalg.eigh. (N, B) → (N, B)"""
    E, phi = torch.linalg.eigh(H)  # (N,), (N, N)
    phi = phi.to(psi0.dtype)
    c = torch.einsum("xn,xb->nb", phi, psi0)  # components in the eigenbasis (phi is real orthogonal)
    return torch.einsum("xn,nb->xb", phi, torch.exp(-1j * E * t)[:, None] * c)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/quantum/schrodinger_crank_nicolson/plot.py
"""Animate the packets sloshing in the anharmonic well, with norm (CN vs explicit Euler) and energy panels."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.ticker import FuncFormatter, LogLocator, NullLocator

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(
    x: torch.Tensor,
    V: torch.Tensor,
    times: torch.Tensor,
    psi: torch.Tensor,
    x_mean: torch.Tensor,
    norms: torch.Tensor,
    energies: torch.Tensor,
    t_euler: torch.Tensor,
    n_euler: torch.Tensor,
    x0s: list[float],
    dt: float,
    path: Path,
) -> Path:
    xs, Vs, t = x.cpu().numpy(), V.cpu().numpy(), times.cpu().numpy()
    prob = (psi.abs() ** 2).cpu().numpy()  # (T, N, B)
    re0 = psi[:, :, 0].real.cpu().numpy()  # (T, N) real part of the first packet
    xm = x_mean.cpu().numpy()  # (T, B)
    nrm, E = norms.cpu().numpy(), energies.cpu().numpy()  # (T, B)
    te, ne = t_euler.cpu().numpy(), n_euler.cpu().numpy()  # (S,), (S, B)
    B = prob.shape[2]
    colors = ACCENT[:B]

    fig = plt.figure(figsize=(8.4, 4.4))
    grid = fig.add_gridspec(2, 2, width_ratios=[1.7, 1])
    ax = fig.add_subplot(grid[:, 0])
    ax_n = fig.add_subplot(grid[0, 1])
    ax_e = fig.add_subplot(grid[1, 1])

    # left: V(x) in energy units; each packet drawn as |ψ|² standing on its own energy level ⟨H⟩
    x_view = x0s[0] + 2.5
    e_top = E[0].max() + 0.35 * (E[0].max() + 1)
    scale = 0.22 * e_top / prob.max()
    re_scale = 0.6 * scale * prob.max() / abs(re0).max()  # Re ψ drawn a bit smaller than |ψ|²
    ax.plot(xs, Vs, color="#6b7280", lw=2, label="V(x)")
    ax.set_xlim(-x_view, x_view)
    ax.set_ylim(0, e_top + 0.08 * e_top)
    ax.set_xlabel("x")
    ax.set_ylabel("energy   (packets: |ψ|² at height ⟨H⟩)")
    lines, dots = [], []
    for b in range(B):
        ax.axhline(E[0, b], color=colors[b], lw=0.6, alpha=0.4)
        (line,) = ax.plot([], [], color=colors[b], lw=2, label=f"|ψ|²,  x0 = {x0s[b]:.1f}")
        (dot,) = ax.plot([], [], "v", color=colors[b], ms=7)
        lines.append(line)
        dots.append(dot)
    (re_line,) = ax.plot([], [], color=colors[0], lw=0.8, ls="--", alpha=0.7, label=f"Re ψ,  x0 = {x0s[0]:.1f}")
    clock = ax.text(0.02, 0.97, "", transform=ax.transAxes, va="top", family="monospace", fontsize=9)
    ax.legend(loc="upper right", fontsize=7)

    # top right: norm against the step number (log–log) — CN stays at 1, explicit Euler explodes
    n_cn = round(t[-1] / dt)
    plain = FuncFormatter(lambda v, _: f"{v:.0e}" if v >= 1e4 else f"{v:g}")  # plain-text ticks (no mathtext)
    ax_n.set_xscale("log")
    ax_n.set_yscale("log")
    ax_n.set_xlim(1, n_cn)
    ax_n.set_ylim(0.3, max(10.0, 3 * ne.max()))
    for axis in (ax_n.xaxis, ax_n.yaxis):
        axis.set_major_locator(LogLocator(base=10 if axis is ax_n.xaxis else 1e4))
        axis.set_minor_locator(NullLocator())
        axis.set_major_formatter(plain)
    steps_e = te / dt  # (S,) step numbers of the Euler run
    n_axis = np.maximum(t / dt, 1)  # (T,) CN step numbers; t = 0 drawn at n = 1 on the log axis
    ax_n.plot(steps_e[1:], ne[1:, 0], "o-", ms=2, color="#dc2626", lw=1.5, label="explicit Euler")
    (norm_line,) = ax_n.plot([], [], color=colors[0], lw=2, label="Crank–Nicolson")
    ax_n.set_title("norm  Σ|ψ|²dx  vs step n", fontsize=9)
    ax_n.legend(loc="center right", fontsize=7)
    ax_n.tick_params(labelsize=7)

    # bottom right: energy of each packet, conserved by CN
    ax_e.set_xlim(0, t[-1])
    ax_e.set_ylim(0, 1.15 * E.max())
    ax_e.set_xlabel("t")
    ax_e.set_title("energy ⟨H⟩ (Crank–Nicolson)", fontsize=9)
    e_lines = [ax_e.plot([], [], color=colors[b], lw=2)[0] for b in range(B)]
    ax_e.tick_params(labelsize=7)

    def update(step: int):
        for b in range(B):
            lines[b].set_data(xs, E[0, b] + scale * prob[step, :, b])
            dots[b].set_data([xm[step, b]], [E[0, b] - 0.03 * e_top])
            e_lines[b].set_data(t[: step + 1], E[: step + 1, b])
        re_line.set_data(xs, E[0, 0] + re_scale * re0[step])
        norm_line.set_data(n_axis[: step + 1], nrm[: step + 1, 0])
        clock.set_text(f"t = {t[step]:5.2f}\n|norm−1| = {abs(nrm[step] - 1).max():.0e}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 72), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'g': 0.05,
    'x0': 2.5,
    't_max': 20.0,
    'dt': 0.005,
    'n_grid': 400
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.quantum.schrodinger_crank_nicolson")]:
    del sys.modules[name]
main = importlib.import_module("problems.quantum.schrodinger_crank_nicolson.main")
problem_dir = Path("problems/quantum/schrodinger_crank_nicolson")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
